# Dates, calendars, and schedules

**Finstack curriculum · Notebook 02**

**Prerequisites:** Work through `01_foundations/core_types_and_money.ipynb` first so you are comfortable importing `finstack_quant`, using `Money`/`Currency`, and running notebooks in this environment.


## Concepts

Cash-flow and accrual math depends on three intertwined ideas:

1. **Day-count conventions** map a calendar span to a *year fraction* (for interest accrual).
2. **Tenors** describe payment spacing (`6M` = semi-annual, `3M` = quarterly).
3. **Schedules** turn a start date, end date, and frequency into concrete payment dates, optionally **adjusted** for weekends and holidays using a **business-day convention** and a **holiday calendar**.

The `finstack_quant.core.dates` module exposes these pieces with stable, Rust-backed implementations. The walkthrough below mirrors the public stubs (`dates.pyi`) and parity tests.


### Day-count conventions (`DayCount`)

`DayCount` enumerates market-standard accrual bases. Use `year_fraction(start, end)` for the accrual between two dates. In finstack, the end date is **exclusive** (same convention as many internal primitives).

`DayCount.calendar_days` counts whole calendar days between dates. `from_name` parses snake-case labels such as `"act_360"`. For example, ACT/360 from `2024-01-01` to `2024-07-01` uses 182 day-basis days in 2024 (leap year), i.e. \(182/360\).

There is no separate `ACT_ACT_ISDA` symbol in the stubs: use `DayCount.ACT_ACT` for ISDA-style actual/actual.


In [ ]:
from datetime import date

from finstack_quant.core.dates import DayCount
print("ACT_360:", DayCount.ACT_360)
print("ACT_365F:", DayCount.ACT_365F)
print("THIRTY_360:", DayCount.THIRTY_360)
print("ACT_ACT (ISDA-style in finstack):", DayCount.ACT_ACT)
yf = DayCount.ACT_360.year_fraction(date(2024, 1, 1), date(2024, 7, 1))
print("ACT/360 year fraction Jan1→Jul1:", round(yf, 6))
cd = DayCount.calendar_days(date(2024, 1, 1), date(2024, 1, 31))
print("Calendar days Jan1→Jan31:", cd)
print("from_name('act_360'):", DayCount.from_name("act_360"))


### Signed accruals

Use `signed_year_fraction` when the sign of the date ordering matters, such as reversing an accrual or validating a cashflow ordering.

In [ ]:
from datetime import date
from finstack_quant.core.dates import DayCount

fwd = DayCount.ACT_360.signed_year_fraction(date(2024, 1, 1), date(2024, 7, 1))
rev = DayCount.ACT_360.signed_year_fraction(date(2024, 7, 1), date(2024, 1, 1))
print("forward signed yf:", round(fwd, 6))
print("reverse signed yf:", round(rev, 6))

### Tenors (`Tenor`)

`Tenor.parse` accepts compact strings (`3M`, `6M`, `1Y`). You can also use helpers like `Tenor.semi_annual()` when you want the canonical object instead of parsing.


In [ ]:
from finstack_quant.core.dates import Tenor
for s in ("3M", "1Y", "6M"):
    t = Tenor.parse(s)
    print(s, "->", repr(t), "count=", t.count, "unit=", t.unit)
print("semi_annual() == parse('6M'):", Tenor.semi_annual() == Tenor.parse("6M"))


### Reporting periods (`PeriodId`, `build_periods`)

`PeriodId` identifies calendar quarters, months, etc. `build_periods` expands range expressions into a `PeriodPlan` of `Period` rows (each with `start`, `end`, and an actual/forecast flag when you pass an actuals cutoff).


In [ ]:
from finstack_quant.core.dates import PeriodId, build_periods
print("month:", PeriodId.month(2024, 6).code)
print("quarter:", PeriodId.quarter(2024, 2).code)
print("annual:", PeriodId.annual(2024).code)
q_all = build_periods("2024Q1..Q4", None)
print("quarters:", [p.id.code for p in q_all.periods])
m_all = build_periods("2024M01..M12", None)
print("months count:", len(m_all.periods), "first:", m_all.periods[0].id.code)
q_af = build_periods("2024Q1..Q4", "2024Q2")
flags = [(p.id.code, p.is_actual) for p in q_af.periods]
print("actuals cutoff Q2:", flags)

# Fiscal periods use a custom year start (e.g. US federal Oct 1)
from finstack_quant.core.dates import FiscalConfig, build_fiscal_periods
fy = build_fiscal_periods("2025Q1..Q2", FiscalConfig.us_federal())
print("fiscal (US) first two:", [p.id.code for p in fy.periods])


### Schedule builder (`ScheduleBuilder`, `StubKind`)

`ScheduleBuilder(start, end)` holds mutable state in Python. Its setters return the same builder, matching Rust's fluent API, so `frequency`, optional `stub_rule`, optional `adjust_with`, and optional `end_of_month` can be chained before `build()` returns a `Schedule` with a `dates` list.


In [ ]:
from datetime import date

from finstack_quant.core.dates import ScheduleBuilder, StubKind, Tenor, BusinessDayConvention
sched = (
    ScheduleBuilder(date(2024, 1, 15), date(2025, 1, 15))
    .frequency(Tenor.parse("3M"))
    .stub_rule(StubKind.NONE)
    .adjust_with(BusinessDayConvention.FOLLOWING, "target2")
    .end_of_month(False)
    .build()
)
print("Schedule:", sched)
print("dates:", [d.isoformat() for d in sched.dates])


### Business-day conventions (`BusinessDayConvention`)

Common roll rules: **following** (next good day), **modified following** (following unless month change, then preceding), **preceding** (previous good day).


In [ ]:
from finstack_quant.core.dates import BusinessDayConvention
print("FOLLOWING:", BusinessDayConvention.FOLLOWING)
print("MODIFIED_FOLLOWING:", BusinessDayConvention.MODIFIED_FOLLOWING)
print("PRECEDING:", BusinessDayConvention.PRECEDING)


### Holiday calendars (`HolidayCalendar`, `adjust`, `available_calendars`)

Resolve a calendar with `HolidayCalendar(code)` or pass a **string code** directly to `adjust`. `available_calendars()` lists registry ids (e.g. `usny`, `nyse`, `target2`).

`adjust` signature is **`adjust(date, convention, calendar)`** — convention before calendar object/code.


In [ ]:
from datetime import date

from finstack_quant.core.dates import (
    HolidayCalendar,
    available_calendars,
    adjust,
    BusinessDayConvention,
)
codes = available_calendars()
print("calendar count:", len(codes))
print("sample:", sorted(c for c in codes if c in ("usny", "nyse", "target2")))
cal = HolidayCalendar("usny")
print("HolidayCalendar:", cal)
d0 = date(2024, 12, 25)
d1 = adjust(d0, BusinessDayConvention.MODIFIED_FOLLOWING, cal)
d2 = adjust(d0, BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
print("Christmas 2024 adjusted:", d0, "->", d1, "(matches string calendar:", d1 == d2, ")")


## Mini-example: semi-annual USD corporate bond

**Setup:** 5-year bond, unadjusted accrual anchors `2024-01-15` → `2029-01-15`, **semi-annual** (`6M`) coupons, **US (`usny`)** holidays, **modified following** adjustment, **30/360** year fractions.

**Accrual vs payment:** For **coupon accrual**, conventions often use **unadjusted** coupon period start/end (the schedule rolled from issue/maturity **before** business-day adjustment) to build year fractions. For **cash payment**, you typically pay on **adjusted** dates (rolled off weekends/holidays). The loop below applies **30/360** between consecutive **adjusted** payment dates, so some fractions differ from one-half when adjustment changes the gap.

On **unadjusted** semi-annual anchors (e.g. 15-Jan → 15-Jul), **30/360** gives **exactly 0.5** for each full six-month accrual period. The code cell prints a few of those pairs after the adjusted schedule.

The first schedule date may move when the anchor falls on a holiday (for example US Martin Luther King Jr. Day in mid-January).


In [ ]:
from datetime import date

from finstack_quant.core.dates import (
    ScheduleBuilder,
    StubKind,
    BusinessDayConvention,
    DayCount,
)
issue = date(2024, 1, 15)
maturity = date(2029, 1, 15)
calendar_id = "usny"

builder = ScheduleBuilder(issue, maturity)
builder.frequency("6M")
builder.stub_rule(StubKind.NONE)
builder.adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, calendar_id)
builder.end_of_month(False)
schedule = builder.build()

dc = DayCount.THIRTY_360
print("Adjusted payment schedule ({} dates):".format(len(schedule.dates)))
for i, d in enumerate(schedule.dates):
    print(f"  {i:2d}  {d.isoformat()}")

print("\n30/360 year fractions between consecutive payment dates:")
for i in range(len(schedule.dates) - 1):
    d0, d1 = schedule.dates[i], schedule.dates[i + 1]
    yf = dc.year_fraction(d0, d1)
    print(f"  {d0.isoformat()}  ->  {d1.isoformat()}   yf={yf:.6f}")

print("\n30/360 on unadjusted semi-annual anchors (accrual-style; each full half-year = 0.5):")
for u0, u1 in (
    (date(2024, 1, 15), date(2024, 7, 15)),
    (date(2024, 7, 15), date(2025, 1, 15)),
    (date(2025, 1, 15), date(2025, 7, 15)),
):
    print(f"  {u0.isoformat()}  ->  {u1.isoformat()}   yf={dc.year_fraction(u0, u1):.6f}")

if schedule.has_warnings():
    print("warnings:", schedule.warnings)


## Takeaways

- **Day-count** choice (`THIRTY_360`, `ACT_360`, `ACT_ACT`, …) controls how calendar time becomes **year fractions**; know whether your `end` date is treated as **exclusive**.
- **Tenors** encode coupon spacing; `Tenor.parse("6M")` is the natural semi-annual knob for `ScheduleBuilder.frequency`.
- **Schedules** combine frequency, stub rules, optional EOM logic, and **business-day adjustment** against a named **holiday calendar**.
- **Period plans** from `build_periods` are ideal for **reporting / actual vs forecast** partitions, separate from cash-flow schedule generation.

Next notebooks can combine these dates with instruments (bonds, swaps) now that calendars and schedules are in place.


## Fiscal periods

`FiscalConfig` and `build_fiscal_periods` let statement models use non-calendar fiscal years without hand-building period boundaries.

In [ ]:
from finstack_quant.core.dates import FiscalConfig, build_fiscal_periods

fiscal = FiscalConfig(4, 1)  # fiscal year starts April 1
plan = build_fiscal_periods("2024Q1..2024Q4", fiscal, actuals_cutoff="2024Q2")
for period in plan.periods:
    print(period.id, period.start, "->", period.end, "actual=" + str(period.is_actual))